# 📊 IntelliCode-SL | Coding SLM — Full Benchmark

Benchmarks 5 configurations on identical test prompts across `debug`, `generate`, and `modify` tasks.

| Config | Model | Precision | Adapter | Notes |
|--------|-------|-----------|---------|-------|
| **A** | Qwen2.5-Coder-3B | fp16 | ❌ None | Base model, no fine-tuning |
| **B** | Qwen2.5-Coder-3B | 4-bit | ✅ fp16 | Fine-tuned IntelliCode adapter |
| **E** | Qwen2.5-Coder-3B | 4-bit | ❌ None | 4-bit base, no fine-tuning |
| **C** | Qwen3-32B | API | ❌ N/A | Groq API — 10x larger |
| **D** | LLaMA 3.3-70B | API | ❌ N/A | Groq API — 23x larger |

> Config E is the control for Config B — isolates the adapter's contribution from quantization alone.
> Run cells top to bottom. GPU runtime required. Groq API key required for C & D.

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q unsloth transformers peft accelerate bitsandbytes groq
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────
import os, torch, time, gc, json
from collections import defaultdict
from google.colab import drive
from huggingface_hub import login
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from groq import Groq

os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

print("✅ Imports done")
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

In [ ]:
# ── Cell 3: Config + Drive + Login ─────────────────────────────
drive.mount("/content/drive")

HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← HuggingFace token
GROQ_API_KEY = "gsk_XXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← Groq API key (console.groq.com)

# Config A uses full precision — different model string from Config B
MODEL_NAME_FP16 = "unsloth/Qwen2.5-Coder-3B-Instruct"        # fp16, no adapter
MODEL_NAME_4BIT = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"  # 4-bit, with adapter

ADAPTER_PATH = "/content/drive/MyDrive/IntelliCode-SL/adapters/coding_adapter"
SAVE_PATH    = "/content/drive/MyDrive/IntelliCode-SL/benchmarks/coding_slm_full_benchmark.json"
MAX_SEQ_LEN  = 2048
TASKS        = ["debug", "generate", "modify"]

login(token=HF_TOKEN)
groq_client = Groq(api_key=GROQ_API_KEY)
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

print("✅ Drive mounted + HF login done + Groq client ready")
print(f"   Config A model : {MODEL_NAME_FP16}")
print(f"   Config B model : {MODEL_NAME_4BIT}")
print(f"   Adapter path   : {ADAPTER_PATH}")


In [ ]:
# ── Cell 4: Prompt Templates ───────────────────────────────────
# Local SLM prompt (matches fine-tuning format exactly)
SLM_TASK_PROMPTS = {
    "debug"    : "Fix the bug in the following code and return the corrected version.",
    "generate" : "Write code based on the following description.",
    "modify"   : "Modify the following code according to the given instruction.",
}

SLM_PROMPT_TEMPLATE = """### Task: {task_instruction}

### Input:
{input}

### Output:
"""

# API prompt (clean natural language for LLMs)
API_SYSTEM_PROMPT = """You are an expert Python developer. Respond with clean, correct, working code only.
Do not include explanations unless explicitly asked. Do not include markdown fences."""

API_USER_TEMPLATES = {
    "debug"    : "Fix the bug in the following code and return only the corrected version:\n\n{input}",
    "generate" : "Write Python code for the following:\n\n{input}",
    "modify"   : "Modify the following code as instructed and return only the modified version:\n\n{input}",
}

print("✅ Prompt templates defined")

In [ ]:
# ── Cell 5: Test Cases (20 per task = 60 total) ────────────────
# Balanced across difficulty: easy, medium, hard
test_cases = {
    "debug": [
        # Easy
        {"input": "def subtract(a, b):\n    return a + b",
         "keywords": ["return a - b"], "difficulty": "easy"},
        {"input": "def is_even(n):\n    return n % 2 == 1",
         "keywords": ["== 0"], "difficulty": "easy"},
        {"input": "def square(n):\n    result = n * n",
         "keywords": ["return"], "difficulty": "easy"},
        {"input": "def get_last(lst):\n    return lst[len(lst)]",
         "keywords": ["len(lst) - 1", "lst[-1]"], "difficulty": "easy"},
        {"input": "def sort_desc(lst):\n    return sorted(lst)",
         "keywords": ["reverse=True"], "difficulty": "easy"},
        # Medium
        {"input": "def factorial(n):\n    if n == 0:\n        return 0\n    return n * factorial(n-1)",
         "keywords": ["return 1"], "difficulty": "medium"},
        {"input": "def avg(nums):\n    return sum(nums) / len(nums)",
         "keywords": ["if not nums", "if len", "ZeroDivision"], "difficulty": "medium"},
        {"input": "def get_name(d):\n    return d['name']\n\nget_name({'age': 25})",
         "keywords": [".get("], "difficulty": "medium"},
        {"input": "def append_item(item, lst=[]):\n    lst.append(item)\n    return lst",
         "keywords": ["None", "lst is None"], "difficulty": "medium"},
        {"input": "total = 0\ndef add(n):\n    total += n\n    return total",
         "keywords": ["global"], "difficulty": "medium"},
        # Hard
        {"input": "def make_counter():\n    count = 0\n    def inc():\n        count += 1\n        return count\n    return inc",
         "keywords": ["nonlocal"], "difficulty": "hard"},
        {"input": "def rm_evens(nums):\n    for n in nums:\n        if n % 2 == 0:\n            nums.remove(n)\n    return nums",
         "keywords": ["comprehension", "if n % 2 != 0", "if n % 2"], "difficulty": "hard"},
        {"input": "def merge(a, b):\n    r = []\n    i = j = 0\n    while i < len(a) or j < len(b):\n        if a[i] < b[j]:\n            r.append(a[i]); i+=1\n        else:\n            r.append(b[j]); j+=1\n    return r",
         "keywords": ["and", "extend", "i < len(a) and"], "difficulty": "hard"},
        {"input": "import threading\ncounter = 0\ndef increment():\n    global counter\n    counter += 1\nthreads = [threading.Thread(target=increment) for _ in range(1000)]\nfor t in threads: t.start()\nfor t in threads: t.join()",
         "keywords": ["Lock", "lock"], "difficulty": "hard"},
        {"input": "def most_freq(lst):\n    counts = {}\n    for x in lst:\n        counts[x] = counts.get(x,0)+1\n    return max(counts)",
         "keywords": ["key=counts.get", "key="], "difficulty": "hard"},
        # Extra medium
        {"input": "age = input('age: ')\nif age > 18:\n    print('adult')",
         "keywords": ["int("], "difficulty": "medium"},
        {"input": "def countdown(n):\n    while n > 0:\n        print(n)",
         "keywords": ["n -= 1"], "difficulty": "easy"},
        {"input": "def read(path):\n    f = open(path, 'r')\n    content = f.read()\n    return content",
         "keywords": ["with open"], "difficulty": "medium"},
        {"input": "def valid_age(age):\n    return age > 0 or age < 150",
         "keywords": ["and"], "difficulty": "easy"},
        {"input": "def safe_div(a, b):\n    try:\n        return a / b\n    except:\n        pass",
         "keywords": ["ZeroDivisionError", "return None", "return 0"], "difficulty": "medium"},
    ],
    "generate": [
        # Easy
        {"input": "Write a function to check if a number is prime",
         "keywords": ["def", "return", "for"], "difficulty": "easy"},
        {"input": "Write a function to reverse a string",
         "keywords": ["def", "return"], "difficulty": "easy"},
        {"input": "Write a function to find the factorial of a number",
         "keywords": ["def", "return", "factorial"], "difficulty": "easy"},
        {"input": "Write a function to check if a string is a palindrome",
         "keywords": ["def", "return"], "difficulty": "easy"},
        {"input": "Write a function to find the maximum element in a list",
         "keywords": ["def", "return"], "difficulty": "easy"},
        # Medium
        {"input": "Write a binary search function on a sorted list",
         "keywords": ["def", "mid", "while", "left", "right"], "difficulty": "medium"},
        {"input": "Write a Stack class with push, pop, peek, and is_empty methods",
         "keywords": ["class", "def push", "def pop", "def peek"], "difficulty": "medium"},
        {"input": "Write a function to flatten a nested list of any depth",
         "keywords": ["def", "isinstance", "extend", "append"], "difficulty": "medium"},
        {"input": "Write a decorator that times how long a function takes to run",
         "keywords": ["def", "wrapper", "time", "wraps"], "difficulty": "medium"},
        {"input": "Write a function to validate an email address using regex",
         "keywords": ["import re", "def", "@"], "difficulty": "medium"},
        # Hard
        {"input": "Write a merge sort implementation",
         "keywords": ["def merge", "def merge_sort", "left", "right"], "difficulty": "hard"},
        {"input": "Implement an LRU cache class with get and put methods",
         "keywords": ["class", "OrderedDict", "def get", "def put"], "difficulty": "hard"},
        {"input": "Write a function to find all permutations of a list",
         "keywords": ["def", "return", "append"], "difficulty": "hard"},
        {"input": "Implement a Trie data structure with insert and search methods",
         "keywords": ["class", "def insert", "def search", "children"], "difficulty": "hard"},
        {"input": "Write Dijkstra's shortest path algorithm",
         "keywords": ["def", "heapq", "dist", "heap"], "difficulty": "hard"},
        # Extra medium
        {"input": "Write a memoize decorator for caching function results",
         "keywords": ["def", "cache", "wrapper"], "difficulty": "medium"},
        {"input": "Write a generator function that yields chunks of a list",
         "keywords": ["def", "yield"], "difficulty": "medium"},
        {"input": "Write a retry decorator with exponential backoff",
         "keywords": ["def", "retry", "sleep", "attempt"], "difficulty": "medium"},
        {"input": "Write a thread-safe singleton class in Python",
         "keywords": ["class", "Lock", "_instance"], "difficulty": "hard"},
        {"input": "Write a function to compute the Levenshtein distance between two strings",
         "keywords": ["def", "dp", "range"], "difficulty": "hard"},
    ],
    "modify": [
        # Easy
        {"input": "Add type hints to this function:\ndef add(a, b):\n    return a + b",
         "keywords": ["->", "int", "float"], "difficulty": "easy"},
        {"input": "Add error handling to this function:\ndef divide(a, b):\n    return a / b",
         "keywords": ["try", "except", "ZeroDivision"], "difficulty": "easy"},
        {"input": "Convert this loop to a list comprehension:\ndef squares(nums):\n    result = []\n    for n in nums:\n        result.append(n**2)\n    return result",
         "keywords": ["[", "**2", "for n in"], "difficulty": "easy"},
        {"input": "Add logging to this function:\ndef process(data):\n    result = [x*2 for x in data]\n    return result",
         "keywords": ["import logging", "logger", "logging"], "difficulty": "easy"},
        {"input": "Refactor to use a dictionary instead of if-elif:\ndef day_name(n):\n    if n==1: return 'Mon'\n    elif n==2: return 'Tue'\n    elif n==3: return 'Wed'\n    else: return 'Unknown'",
         "keywords": ["{", ".get(", "dict"], "difficulty": "easy"},
        # Medium
        {"input": "Add caching to this function:\ndef fib(n):\n    if n <= 1: return n\n    return fib(n-1) + fib(n-2)",
         "keywords": ["lru_cache", "cache", "memo"], "difficulty": "medium"},
        {"input": "Make this class thread-safe:\nclass Counter:\n    def __init__(self):\n        self.val = 0\n    def inc(self):\n        self.val += 1",
         "keywords": ["Lock", "lock", "threading"], "difficulty": "medium"},
        {"input": "Add context manager support to this class:\nclass DB:\n    def __init__(self):\n        self.conn = None\n    def connect(self): pass\n    def disconnect(self): pass",
         "keywords": ["__enter__", "__exit__"], "difficulty": "medium"},
        {"input": "Add input validation to this function:\ndef get_elem(lst, idx):\n    return lst[idx]",
         "keywords": ["raise", "isinstance", "if"], "difficulty": "medium"},
        {"input": "Optimize this O(n^2) function to O(n):\ndef has_dups(lst):\n    for i in range(len(lst)):\n        for j in range(i+1, len(lst)):\n            if lst[i]==lst[j]: return True\n    return False",
         "keywords": ["set"], "difficulty": "medium"},
        # Hard
        {"input": "Convert to async:\ndef fetch(url):\n    import requests\n    return requests.get(url).json()",
         "keywords": ["async", "await", "aiohttp"], "difficulty": "hard"},
        {"input": "Add retry logic with exponential backoff to this API call:\ndef call_api(url):\n    import requests\n    return requests.get(url).json()",
         "keywords": ["retry", "sleep", "attempt", "backoff"], "difficulty": "hard"},
        {"input": "Refactor this class to use properties:\nclass Circle:\n    def __init__(self, r):\n        self.radius = r\n        self.area = 3.14 * r**2",
         "keywords": ["@property", "def radius", "def area"], "difficulty": "hard"},
        {"input": "Add observer pattern support to this class:\nclass Inventory:\n    def __init__(self):\n        self.items = {}\n    def add(self, name, qty):\n        self.items[name] = self.items.get(name,0) + qty",
         "keywords": ["observer", "subscribe", "notify", "listener"], "difficulty": "hard"},
        {"input": "Add circuit breaker pattern to this client:\nclass Client:\n    def __init__(self, url):\n        self.url = url\n    def call(self, endpoint):\n        import requests\n        return requests.get(f'{self.url}/{endpoint}').json()",
         "keywords": ["circuit", "failure", "open", "closed", "state"], "difficulty": "hard"},
        # Extra medium
        {"input": "Refactor to use generators:\ndef read_file(path):\n    with open(path) as f:\n        lines = f.readlines()\n    return [l.strip() for l in lines]",
         "keywords": ["yield"], "difficulty": "medium"},
        {"input": "Add method chaining by returning self from each method:\nclass QueryBuilder:\n    def __init__(self, table):\n        self.table = table\n        self.conditions = []\n    def where(self, c):\n        self.conditions.append(c)\n    def build(self):\n        return f'SELECT * FROM {self.table}'",
         "keywords": ["return self"], "difficulty": "medium"},
        {"input": "Add soft delete support to this repository:\nclass Repo:\n    def __init__(self, db):\n        self.db = db\n    def delete(self, id):\n        self.db.execute('DELETE FROM users WHERE id=?',(id,))",
         "keywords": ["deleted_at", "is_active", "UPDATE"], "difficulty": "medium"},
        {"input": "Add environment variable configuration to this class:\nclass Config:\n    HOST = 'localhost'\n    PORT = 8080\n    DEBUG = False",
         "keywords": ["os.getenv", "os.environ"], "difficulty": "medium"},
        {"input": "Add serialization to this model:\nclass User:\n    def __init__(self, id, name, email):\n        self.id = id\n        self.name = name\n        self.email = email",
         "keywords": ["to_dict", "from_dict", "json"], "difficulty": "medium"},
    ]
}

print(f"✅ Test cases ready")
for task, cases in test_cases.items():
    easy   = sum(1 for c in cases if c['difficulty'] == 'easy')
    medium = sum(1 for c in cases if c['difficulty'] == 'medium')
    hard   = sum(1 for c in cases if c['difficulty'] == 'hard')
    print(f"   {task:<10}: {len(cases)} cases  (easy={easy}, medium={medium}, hard={hard})")

In [ ]:
# ── Cell 6: Scoring Function ───────────────────────────────────
def score_output(output, case):
    """
    Returns a score dict with:
    - valid  : output is non-empty and looks like code
    - keyword: at least one expected keyword found
    - score  : 0.0 to 1.0 (0=fail, 0.5=valid but no keyword, 1.0=full pass)
    """
    output = output.strip()

    # Validity check: non-empty, has some code content
    valid = (
        len(output) > 20 and
        not output.startswith("###") and
        not output.startswith("I cannot") and
        not output.startswith("I don't") and
        any(kw in output for kw in ["def ", "class ", "return", "import", "for ", "while ", "if ", "="])
    )

    # Keyword check: at least one expected keyword present
    keywords_found = [kw for kw in case["keywords"] if kw in output]
    keyword_pass   = len(keywords_found) > 0

    if valid and keyword_pass:
        score = 1.0
    elif valid:
        score = 0.5
    else:
        score = 0.0

    return {
        "valid": valid,
        "keyword_pass": keyword_pass,
        "keywords_found": keywords_found,
        "score": score
    }

print("✅ Scoring function defined")

In [ ]:
# ── Cell 7: Local SLM Inference Helper ────────────────────────
def run_local_inference(model, tokenizer, task, input_text, max_new_tokens=300):
    prompt = SLM_PROMPT_TEMPLATE.format(
        task_instruction = SLM_TASK_PROMPTS[task],
        input            = input_text
    )
    inputs = tokenizer(
        prompt,
        return_tensors = "pt",
        truncation     = True,
        max_length     = MAX_SEQ_LEN
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = None,   # must be None when do_sample=False
            top_p          = None,   # must be None when do_sample=False
            pad_token_id   = tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Output:")[-1].strip()

print("✅ Local inference helper defined")


In [ ]:
# ── Cell 8: Groq API Inference Helper ─────────────────────────
def run_api_inference(task, input_text, model_name, max_retries=3):
    user_msg = API_USER_TEMPLATES[task].format(input=input_text)
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model    = model_name,
                messages = [
                    {"role": "system", "content": API_SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                max_tokens  = 600,
                temperature = 0.0,
            )
            output = response.choices[0].message.content.strip()
            # Strip markdown code fences if present
            if output.startswith("```"):
                lines = output.split("\n")
                output = "\n".join(lines[1:-1] if lines[-1] == "```" else lines[1:])
            return output
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"    ⚠️  API error after {max_retries} attempts: {e}")
                return ""
            time.sleep(2 ** attempt)

print("✅ Groq API inference helper defined")

In [ ]:
# ── Cell 9: Evaluation Runner ──────────────────────────────────
def evaluate_local(model, tokenizer, config_name):
    print(f"\n{'='*60}")
    print(f"  Evaluating: {config_name}")
    print(f"{'='*60}")

    all_results = []
    per_task    = {t: {"scores": [], "times": [], "valid": 0, "kw_pass": 0} for t in TASKS}
    total_time  = 0
    idx         = 0

    for task, cases in test_cases.items():
        print(f"\n  [{task.upper()}] running {len(cases)} cases...")
        for case in cases:
            t0      = time.time()
            output  = run_local_inference(model, tokenizer, task, case["input"])
            elapsed = time.time() - t0
            total_time += elapsed

            result = score_output(output, case)
            per_task[task]["scores"].append(result["score"])
            per_task[task]["times"].append(elapsed)
            if result["valid"]:        per_task[task]["valid"]   += 1
            if result["keyword_pass"]: per_task[task]["kw_pass"] += 1

            all_results.append({
                "task": task, "difficulty": case["difficulty"],
                "input": case["input"][:80],
                "output": output[:300],
                "score": result["score"],
                "valid": result["valid"],
                "keyword_pass": result["keyword_pass"],
                "time_ms": elapsed * 1000
            })
            idx += 1

    return _summarize(config_name, per_task, all_results, total_time)

def evaluate_api(api_model_name, config_name):
    print(f"\n{'='*60}")
    print(f"  Evaluating: {config_name}")
    print(f"{'='*60}")

    all_results = []
    per_task    = {t: {"scores": [], "times": [], "valid": 0, "kw_pass": 0} for t in TASKS}
    total_time  = 0

    for task, cases in test_cases.items():
        print(f"\n  [{task.upper()}] running {len(cases)} cases...")
        for case in cases:
            t0      = time.time()
            output  = run_api_inference(task, case["input"], api_model_name)
            elapsed = time.time() - t0
            total_time += elapsed
            time.sleep(0.3)  # Avoid rate limiting

            result = score_output(output, case)
            per_task[task]["scores"].append(result["score"])
            per_task[task]["times"].append(elapsed)
            if result["valid"]:        per_task[task]["valid"]   += 1
            if result["keyword_pass"]: per_task[task]["kw_pass"] += 1

            all_results.append({
                "task": task, "difficulty": case["difficulty"],
                "input": case["input"][:80],
                "output": output[:300],
                "score": result["score"],
                "valid": result["valid"],
                "keyword_pass": result["keyword_pass"],
                "time_ms": elapsed * 1000
            })

    return _summarize(config_name, per_task, all_results, total_time)

def _summarize(config_name, per_task, all_results, total_time):
    total_cases = sum(len(v["scores"]) for v in per_task.values())
    total_score = sum(s for v in per_task.values() for s in v["scores"])
    avg_score   = total_score / total_cases * 100
    avg_ms      = total_time / total_cases * 1000

    full_pass   = sum(1 for r in all_results if r["score"] == 1.0)
    partial     = sum(1 for r in all_results if r["score"] == 0.5)
    fail        = sum(1 for r in all_results if r["score"] == 0.0)

    print(f"\n  ✅ Avg Score     : {avg_score:.1f}%  (full={full_pass}, partial={partial}, fail={fail})")
    print(f"  ⏱  Avg Latency   : {avg_ms:.0f} ms/prompt")
    print(f"\n  {'Task':<12} {'Avg Score':>10} {'Valid':>7} {'KW Pass':>8} {'Avg ms':>8}")
    print(f"  {'-'*48}")
    for task in TASKS:
        s        = per_task[task]
        n        = len(s["scores"])
        avg_sc   = sum(s["scores"]) / n * 100
        avg_t    = sum(s["times"]) / n * 1000
        print(f"  {task:<12} {avg_sc:>9.1f}% {s['valid']:>7} {s['kw_pass']:>8} {avg_t:>7.0f}ms")

    return {
        "config": config_name,
        "avg_score": avg_score,
        "full_pass": full_pass,
        "partial": partial,
        "fail": fail,
        "total": total_cases,
        "avg_time_ms": avg_ms,
        "per_task": {
            task: {
                "avg_score": sum(per_task[task]["scores"]) / len(per_task[task]["scores"]) * 100,
                "valid": per_task[task]["valid"],
                "kw_pass": per_task[task]["kw_pass"],
                "total": len(per_task[task]["scores"]),
                "avg_time_ms": sum(per_task[task]["times"]) / len(per_task[task]["times"]) * 1000
            } for task in TASKS
        },
        "samples": all_results
    }

print("✅ Evaluation runners defined")

In [ ]:
# ── Cell 10: Config A — Base Model fp16, No Adapter ────────────
print("Loading Config A: Base model fp16, NO adapter...")

model_A, tokenizer_A = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME_FP16,   # full precision model
    max_seq_length = MAX_SEQ_LEN,
    dtype          = torch.float16,
    load_in_4bit   = False,
    token          = HF_TOKEN,
)
FastLanguageModel.for_inference(model_A)
model_A.eval()

print("✅ Config A loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── Cell 11: Evaluate Config A ─────────────────────────────────
results_A = evaluate_local(
    model_A, tokenizer_A,
    "Config A: Base Model fp16 (no fine-tuning)"
)

In [ ]:
# ── Cell 12: Unload Config A ───────────────────────────────────
del model_A, tokenizer_A
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Config A unloaded — VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 13: Config B — 4-bit Base + fp16 Adapter ──────────────
print("Loading Config B: 4-bit base + fp16 adapter (fine-tuned)...")

model_B, tokenizer_B = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME_4BIT,   # 4-bit quantized model
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
model_B = PeftModel.from_pretrained(
    model_B,
    ADAPTER_PATH,
    torch_dtype = torch.float16
)
FastLanguageModel.for_inference(model_B)
model_B.eval()

print("✅ Config B loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── Cell 14: Evaluate Config B ─────────────────────────────────
results_B = evaluate_local(
    model_B, tokenizer_B,
    "Config B: 4-bit Base + fp16 Adapter (IntelliCode fine-tuned)"
)

In [ ]:
# ── Cell 15: Unload Config B ───────────────────────────────────
del model_B, tokenizer_B
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Config B unloaded — VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 16: Config E — 4-bit Base, No Adapter ─────────────────
print("Loading Config E: 4-bit base model, NO adapter (no fine-tuning)...")

model_E, tokenizer_E = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME_4BIT,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
FastLanguageModel.for_inference(model_E)
model_E.eval()

print("✅ Config E loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── Cell 17: Evaluate Config E ─────────────────────────────────
results_E = evaluate_local(
    model_E, tokenizer_E,
    "Config E: 4-bit Base (no fine-tuning)"
)


In [ ]:
# ── Cell 18: Unload Config E ───────────────────────────────────
del model_E, tokenizer_E
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Config E unloaded — VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── Cell 19: Config C — Qwen3-32B via Groq API ─────────────────
# Verify latest model ID at: https://console.groq.com/docs/models
print("Evaluating Config C: Qwen3-32B via Groq API...")
print("   (API calls — no GPU needed, expect ~2-5s per call)")

results_C = evaluate_api(
    api_model_name = "qwen/qwen3-32b",
    config_name    = "Config C: Qwen3-32B (Groq API)"
)


In [ ]:
# ── Cell 17: Config D — LLaMA 3.3-70B via Groq API ─────────────
print("Evaluating Config D: LLaMA 3.3-70B via Groq API...")
print("   (API calls — expect ~30-60s per task)")

results_D = evaluate_api(
    api_model_name = "llama-3.3-70b-versatile",
    config_name    = "Config D: LLaMA 3.3-70B (Groq API)"
)

In [ ]:
# ── Cell 21: Final Report ──────────────────────────────────────
all_results = {
    "A": results_A,
    "B": results_B,
    "E": results_E,
    "C": results_C,
    "D": results_D,
}

labels = {
    "A": "Base fp16 (no FT)",
    "B": "4-bit + Adapter (FT)",
    "E": "4-bit Base (no FT)",
    "C": "Qwen3-32B (API)",
    "D": "LLaMA3.3-70B (API)",
}

print("\n" + "="*78)
print("  FULL BENCHMARK RESULTS — IntelliCode Coding SLM (5 configs)")
print("="*78)

print(f"\n  {'Config':<26} {'Avg Score':>10} {'Full Pass':>10} {'Partial':>8} {'Fail':>6} {'Latency':>10}")
print(f"  {'-'*72}")
for k, r in all_results.items():
    print(f"  {labels[k]:<26} {r['avg_score']:>9.1f}% {r['full_pass']:>10} {r['partial']:>8} {r['fail']:>6} {r['avg_time_ms']:>8.0f}ms")

print(f"\n  Per-Task Average Score:")
print(f"  {'Task':<10}" + "".join(f"{labels[k]:>20}" for k in all_results))
print(f"  {'-'*110}")
for task in TASKS:
    row = f"  {task:<10}"
    for k in all_results:
        score = all_results[k]['per_task'][task]['avg_score']
        row  += f"{score:>19.1f}%"
    print(row)

# Key comparisons
ft_gain_vs_fp16  = results_B['avg_score'] - results_A['avg_score']
ft_gain_vs_4bit  = results_B['avg_score'] - results_E['avg_score']
print(f"\n  Adapter gain (B vs A fp16)   : {ft_gain_vs_fp16:+.1f}%")
print(f"  Adapter gain (B vs E 4-bit)  : {ft_gain_vs_4bit:+.1f}%  ← pure adapter contribution")
print(f"  vs Qwen3-32B     (B vs C)    : {results_B['avg_score'] - results_C['avg_score']:+.1f}%")
print(f"  vs LLaMA3.3-70B  (B vs D)    : {results_B['avg_score'] - results_D['avg_score']:+.1f}%")

print(f"\n{'='*78}")
print(f"  KEY INSIGHTS")
print(f"{'='*78}")
if results_B['avg_score'] > results_E['avg_score']:
    print(f"  ✅ Adapter adds +{ft_gain_vs_4bit:.1f}% over 4-bit base alone.")
else:
    print(f"  ℹ️  4-bit base without adapter scores {results_E['avg_score']:.1f}% — adapter did not help on these tasks.")
if results_B['avg_score'] > results_C['avg_score']:
    print(f"  ✅ Fine-tuned 3B outperforms Qwen3-32B (10x larger) on domain tasks.")
elif results_B['avg_score'] >= results_C['avg_score'] - 5:
    print(f"  ✅ Fine-tuned 3B within 5% of Qwen3-32B (10x larger model).")
if results_B['avg_score'] > results_D['avg_score']:
    print(f"  ✅ Fine-tuned 3B outperforms LLaMA3.3-70B (23x larger) on domain tasks.")
elif results_B['avg_score'] >= results_D['avg_score'] - 5:
    print(f"  ✅ Fine-tuned 3B within 5% of LLaMA3.3-70B (23x larger model).")
print(f"{'='*78}")


In [ ]:
# ── Cell 22: Per-Difficulty Breakdown ──────────────────────────
print("\n  Per-Difficulty Score Breakdown:")
for difficulty in ["easy", "medium", "hard"]:
    print(f"\n  [{difficulty.upper()}]")
    print(f"  {'Config':<26} {'Score':>10}")
    print(f"  {'-'*38}")
    for k, r in all_results.items():
        diff_samples = [s for s in r["samples"] if s["difficulty"] == difficulty]
        if diff_samples:
            avg = sum(s["score"] for s in diff_samples) / len(diff_samples) * 100
            print(f"  {labels[k]:<26} {avg:>9.1f}%")


In [ ]:
# ── Cell 23: Save Full Results to Drive ────────────────────────
output = {
    "benchmark"           : "Coding SLM — Full 5-Config Benchmark",
    "test_cases_per_task" : {t: len(c) for t, c in test_cases.items()},
    "configs": {
        "A": {"label": labels["A"], "avg_score": results_A["avg_score"],
              "full_pass": results_A["full_pass"], "partial": results_A["partial"],
              "fail": results_A["fail"], "total": results_A["total"],
              "avg_time_ms": results_A["avg_time_ms"], "per_task": results_A["per_task"]},
        "B": {"label": labels["B"], "avg_score": results_B["avg_score"],
              "full_pass": results_B["full_pass"], "partial": results_B["partial"],
              "fail": results_B["fail"], "total": results_B["total"],
              "avg_time_ms": results_B["avg_time_ms"], "per_task": results_B["per_task"]},
        "E": {"label": labels["E"], "avg_score": results_E["avg_score"],
              "full_pass": results_E["full_pass"], "partial": results_E["partial"],
              "fail": results_E["fail"], "total": results_E["total"],
              "avg_time_ms": results_E["avg_time_ms"], "per_task": results_E["per_task"]},
        "C": {"label": labels["C"], "avg_score": results_C["avg_score"],
              "full_pass": results_C["full_pass"], "partial": results_C["partial"],
              "fail": results_C["fail"], "total": results_C["total"],
              "avg_time_ms": results_C["avg_time_ms"], "per_task": results_C["per_task"]},
        "D": {"label": labels["D"], "avg_score": results_D["avg_score"],
              "full_pass": results_D["full_pass"], "partial": results_D["partial"],
              "fail": results_D["fail"], "total": results_D["total"],
              "avg_time_ms": results_D["avg_time_ms"], "per_task": results_D["per_task"]},
    }
}

with open(SAVE_PATH, "w") as f:
    json.dump(output, f, indent=2)

assert os.path.exists(SAVE_PATH), f"❌ Save failed — file not found at {SAVE_PATH}"
print(f"✅ Full benchmark results saved → {SAVE_PATH}")
print(f"   File size: {os.path.getsize(SAVE_PATH)/1024:.1f} KB")
